# Data Scrubbing (Fairness through Blindness)

Mask sensitive words (city names, age indicators) before feeding to model.
If the model can't see the word "Москва", it can't rely on it.

In [1]:
import os, json, re, numpy as np, pandas as pd, torch, joblib
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
np.random.seed(42); torch.manual_seed(42)

/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_DIR = Path("..")
df_train = pd.read_csv(BASE_DIR / "data/processed/train.csv")
df_val = pd.read_csv(BASE_DIR / "data/processed/val.csv")
df_test = pd.read_csv(BASE_DIR / "data/processed/test.csv")

mapping = pd.read_csv(BASE_DIR / "data/processed/label_to_supercategory_v1.csv")
label_to_supercat = dict(zip(mapping["label"], mapping["supercategory"]))
for df in [df_train, df_val, df_test]: 
    df["supercategory"] = df["label"].map(label_to_supercat)

le = LabelEncoder()
df_train["y"] = le.fit_transform(df_train["supercategory"])
df_val["y"] = le.transform(df_val["supercategory"])
df_test["y"] = le.transform(df_test["supercategory"])
num_labels = len(le.classes_)
print(f"Train: {len(df_train)}, Labels: {num_labels}")

Train: 16530, Labels: 9


In [3]:
# Define sensitive words to scrub
CITY_WORDS = [
    # Major cities
    "москва", "московская", "московский", "мск",
    "санкт-петербург", "петербург", "спб", "питер", "ленинград",
    "новосибирск", "екатеринбург", "казань", "нижний новгород",
    "челябинск", "самара", "омск", "ростов-на-дону", "уфа",
    "красноярск", "воронеж", "пермь", "волгоград",
    # Medium cities
    "краснодар", "саратов", "тюмень", "тольятти", "ижевск",
    "барнаул", "ульяновск", "иркутск", "хабаровск", "ярославль",
    "владивосток", "махачкала", "томск", "оренбург", "кемерово",
    "новокузнецк", "рязань", "астрахань", "пенза", "липецк",
    "калининград", "тула", "курск", "ставрополь", "сочи",
    # Other
    "минск", "алматы", "киев", "симферополь",
    # Regions
    "область", "край", "республика", "регион",
    "забайкальский", "приморский", "краснодарский",
]

AGE_WORDS = [
    "пенсионер", "пенсионерка", "пенсия", "пенсионный",
    "студент", "студентка", "выпускник", "выпускница",
    "молодой", "молодая", "junior", "senior",
]

ALL_SENSITIVE = set(CITY_WORDS + AGE_WORDS)
print(f"Sensitive words: {len(ALL_SENSITIVE)}")

Sensitive words: 70


In [4]:
# Scrubbing function
def scrub_text(text, mask_token="[MASK]"):
    """Replace sensitive words with mask token"""
    text_lower = text.lower()
    result = text
    
    for word in ALL_SENSITIVE:
        # Case-insensitive replacement
        pattern = re.compile(re.escape(word), re.IGNORECASE)
        result = pattern.sub(mask_token, result)
    
    return result

# Test
test_text = "Опыт работы в Москве, компания в Санкт-Петербурге. Молодой специалист."
print(f"Original: {test_text}")
print(f"Scrubbed: {scrub_text(test_text)}")

Original: Опыт работы в Москве, компания в Санкт-Петербурге. Молодой специалист.
Scrubbed: Опыт работы в Москве, компания в [MASK]е. [MASK] специалист.


In [5]:
# Apply scrubbing to all datasets
print("Scrubbing texts...")
df_train["resume_text_scrubbed"] = df_train["resume_text"].apply(scrub_text)
df_val["resume_text_scrubbed"] = df_val["resume_text"].apply(scrub_text)
df_test["resume_text_scrubbed"] = df_test["resume_text"].apply(scrub_text)

# Count how many texts were modified
modified = (df_train["resume_text"] != df_train["resume_text_scrubbed"]).sum()
print(f"Modified {modified}/{len(df_train)} training texts ({100*modified/len(df_train):.1f}%)")

Scrubbing texts...
Modified 14428/16530 training texts (87.3%)


In [6]:
# Sample weights (sqrt reweighting)
city_counts = df_train["city_group"].value_counts()
raw_w = 1.0 / np.sqrt(city_counts)
city_weight_map = (raw_w / raw_w.mean()).to_dict()
df_train["sample_weight"] = df_train["city_group"].map(city_weight_map).astype(float)
print(f"Weight range: {df_train['sample_weight'].min():.3f} - {df_train['sample_weight'].max():.3f}")

Weight range: 0.148 - 1.470


In [7]:
# Tokenize scrubbed texts
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(b): 
    return tokenizer(b["resume_text_scrubbed"], padding="max_length", truncation=True, max_length=128)

train_ds = Dataset.from_pandas(df_train[["resume_text_scrubbed", "y", "sample_weight"]]).map(tokenize, batched=True)
val_ds = Dataset.from_pandas(df_val[["resume_text_scrubbed", "y"]]).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(df_test[["resume_text_scrubbed", "y"]]).map(tokenize, batched=True)

train_ds = train_ds.rename_column("y", "labels")
val_ds = val_ds.rename_column("y", "labels")
test_ds = test_ds.rename_column("y", "labels")

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels", "sample_weight"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map: 100%|██████████| 5510/5510 [00:07<00:00, 698.55 examples/s]


In [8]:
# Weighted trainer
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        sample_weight = inputs.pop("sample_weight", None)
        labels = inputs["labels"]
        outputs = model(**inputs)
        logits = outputs.logits
        
        loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
        loss_per_sample = loss_fct(logits, labels)
        
        if sample_weight is not None:
            loss_per_sample = loss_per_sample * sample_weight.to(loss_per_sample.dtype)
        
        loss = loss_per_sample.mean()
        return (loss, outputs) if return_outputs else loss

In [9]:
# Train
MODEL_NAME = "bert_scrubbing"

args = TrainingArguments(
    output_dir=f"./models/{MODEL_NAME}", learning_rate=2e-5,
    per_device_train_batch_size=8, per_device_eval_batch_size=8,
    num_train_epochs=2, weight_decay=0.01, logging_steps=100,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="accuracy", 
    remove_unused_columns=False
)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"accuracy": accuracy_score(p.label_ids, preds), 
            "macro_f1": f1_score(p.label_ids, preds, average="macro")}

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)
trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds, 
                          eval_dataset=val_ds, compute_metrics=compute_metrics)

print(f"Training: Scrubbing + sqrt_rw, 2 epochs")
trainer.train()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1614.25it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those p

Training: Scrubbing + sqrt_rw, 2 epochs


/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.540174,1.243614,0.549909,0.566095
2,0.486174,1.108543,0.596189,0.617355


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.50s/it]
/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.38s/it]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.La

TrainOutput(global_step=4134, training_loss=0.6170809789158379, metrics={'train_runtime': 9259.9091, 'train_samples_per_second': 3.57, 'train_steps_per_second': 0.446, 'total_flos': 2174749547351040.0, 'train_loss': 0.6170809789158379, 'epoch': 2.0})

In [10]:
# Evaluate on test
preds = trainer.predict(test_ds)
y_true, y_pred = preds.label_ids, np.argmax(preds.predictions, axis=1)
print(f"\nTEST: Acc={accuracy_score(y_true, y_pred):.4f}, F1={f1_score(y_true, y_pred, average='macro'):.4f}")

/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)



TEST: Acc=0.5942, F1=0.6106


In [11]:
# Fairness evaluation
df_eval = df_test.copy()
df_eval["y_true"], df_eval["y_pred"] = y_true, y_pred

def ovr_rates(df, grp, nc):
    groups = sorted(df[grp].dropna().unique())
    tpr, support = np.zeros((len(groups), nc)), np.zeros((len(groups), nc))
    for gi, g in enumerate(groups):
        dg = df[df[grp] == g]
        yt, yp = dg["y_true"].values, dg["y_pred"].values
        for c in range(nc):
            pm = (yt == c)
            TP, FN = np.sum((yp == c) & pm), np.sum((yp != c) & pm)
            support[gi, c] = pm.sum()
            tpr[gi, c] = TP / (TP + FN) if (TP + FN) > 0 else np.nan
    return tpr, support

def gaps(tpr, sup, ms=30):
    g = []
    for c in range(tpr.shape[1]):
        col = tpr[sup[:, c] >= ms, c]
        col = col[~np.isnan(col)]
        g.append(col.max() - col.min() if len(col) >= 2 else np.nan)
    g = np.array(g); v = g[~np.isnan(g)]
    return v.max() if len(v) else np.nan, v.mean() if len(v) else np.nan

tpr, sup = ovr_rates(df_eval, "city_group", num_labels)
tw, tm = gaps(tpr, sup, 30)
print(f"FAIRNESS (robust): worst={tw:.4f}, macro={tm:.4f}")
print(f"Compare: baseline=0.609 acc, 0.329 worst, 0.116 macro")

FAIRNESS (robust): worst=0.3000, macro=0.1121
Compare: baseline=0.609 acc, 0.329 worst, 0.116 macro


In [12]:
# Save
SAVE_DIR = Path("models") / MODEL_NAME
SAVE_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(SAVE_DIR, safe_serialization=True)
tokenizer.save_pretrained(SAVE_DIR)
joblib.dump(le, SAVE_DIR / "label_encoder.joblib")

# Save scrubbing config
cfg = {
    "method": "Data Scrubbing + sqrt_rw",
    "scrubbed_words": list(ALL_SENSITIVE),
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
    "tpr_gap_worst_robust": float(tw),
    "tpr_gap_macro_robust": float(tm)
}
with open(SAVE_DIR / "training_config.json", "w") as f: 
    json.dump(cfg, f, indent=2, ensure_ascii=False)

print(f"Saved: {SAVE_DIR}")
print(f"\n{'='*50}")
print(f"SUMMARY: Data Scrubbing")
print(f"{'='*50}")
print(f"Scrubbed words: {len(ALL_SENSITIVE)}")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1: {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"TPR Gap (worst): {tw:.4f}")
print(f"TPR Gap (macro): {tm:.4f}")
print(f"{'='*50}")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.16s/it]

Saved: models/bert_scrubbing

SUMMARY: Data Scrubbing
Scrubbed words: 70
Accuracy: 0.5942
Macro F1: 0.6106
TPR Gap (worst): 0.3000
TPR Gap (macro): 0.1121
